# Walmart Sales — Data Cleaning
**Author:** Seema Kumari 

**Dataset:** 561,000 sales records | 5 tables | 2020–2023  
**Purpose:** Profile, clean, validate and export 5 clean CSVs ready for PostgreSQL + Power BI


### Issues found and fixed:
| Table | Issue | Fix Applied |
|---|---|---|
| Customer | 80 duplicate IDs, 87 null City/State | Dedup on ID, fill Unknown |
| Product | 25 duplicate IDs, inconsistent Sub_Category | Dedup, standardize text |
| Store | 2 duplicate IDs, null City/State, mixed State case | Dedup, UPPER State |
| Employee | 10 duplicate IDs, null Position/Dept, 3 date formats | Dedup, fill Unknown, parse dates |
| Sales | 11,000 duplicates, 44K null Shipping, 3 date formats, 12 Payment variants | Full clean |

In [1]:
import pandas as pd
import numpy as np
import warnings
print('Libraries loaded')

Libraries loaded


# Customer Table

In [2]:
customer = pd.read_csv('Customer_Dim.csv')
print(f'Shape: {customer.shape}')
customer.head()

Shape: (1080, 8)


,Customer_ID,Customer_Name,Gender,Age,City,State,Country,Segment
0,1,William Davis,Female,60,Los Angeles,CA,USA,Home Office
1,2,Jane Rodriguez,Female,55,San Jose,ca,USA,Consumer
2,3,Emily Williams,Male,53,San Antonio,TX,USA,Home Office
3,4,James Williams,Male,37,Dallas,TX,USA,Consumer
4,5,Michael Davis,Female,46,philadelphia,PA,USA,Corporate


In [3]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1080 entries, 0 to 1079
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Customer_ID    1080 non-null   int64 
 1   Customer_Name  1080 non-null   object
 2   Gender         1080 non-null   object
 3   Age            1080 non-null   int64 
 4   City           993 non-null    object
 5   State          994 non-null    object
 6   Country        1080 non-null   object
 7   Segment        1080 non-null   object
dtypes: int64(2), object(6)
memory usage: 67.6+ KB


In [4]:
customer.describe()

,Customer_ID,Age
count,1080.000000,1080.000000
mean,500.162963,42.884259
std,288.683499,15.306402
min,1.000000,18.000000
25%,251.750000,29.000000
50%,503.500000,43.000000
75%,747.250000,57.000000
max,1000.000000,69.000000


In [5]:
print(customer.isnull().sum())
print(f'\nNull %:')
print((customer.isnull().sum() / len(customer) * 100).round(2))

Customer_ID       0
Customer_Name     0
Gender            0
Age               0
City             87
State            86
Country           0
Segment           0
dtype: int64

Null %:
Customer_ID      0.00
Customer_Name    0.00
Gender           0.00
Age              0.00
City             8.06
State            7.96
Country          0.00
Segment          0.00
dtype: float64


In [6]:
print(f'Full-row duplicates  : {customer.duplicated().sum()}')
print(f'Customer_ID duplicates: {customer["Customer_ID"].duplicated().sum()}')

Full-row duplicates  : 80
Customer_ID duplicates: 80


In [7]:
customer['City']  = customer['City'].fillna('Unknown')
customer['State'] = customer['State'].fillna('Unknown')

In [8]:
text_cols = ['Customer_Name', 'Gender', 'City', 'Country', 'Segment']
for col in text_cols:
    customer[col] = customer[col].astype(str).str.strip().str.title()

In [9]:
customer['State'] = customer['State'].str.strip().str.upper()

In [10]:
customer = customer.drop_duplicates(subset='Customer_ID', keep='first')

In [11]:
print('Age range:', customer['Age'].min(), '–', customer['Age'].max())
print('Gender values:', customer['Gender'].unique())
print('Segment values:', customer['Segment'].unique())
print(f'\nClean shape: {customer.shape}')

Age range: 18 – 69
Gender values: ['Female' 'Male' 'Other']
Segment values: ['Home Office' 'Consumer' 'Corporate']

Clean shape: (1000, 8)


In [12]:
print('Nulls remaining:', customer.isnull().sum().sum())
print('ID duplicates  :', customer['Customer_ID'].duplicated().sum())
print('State sample   :', customer['State'].unique()[:8])
customer.head(3)

Nulls remaining: 0
ID duplicates  : 0
State sample   : ['CA' 'TX' 'PA' 'AZ' 'IL' 'UNKNOWN' 'NY']


,Customer_ID,Customer_Name,Gender,Age,City,State,Country,Segment
0,1,William Davis,Female,60,Los Angeles,CA,Usa,Home Office
1,2,Jane Rodriguez,Female,55,San Jose,CA,Usa,Consumer
2,3,Emily Williams,Male,53,San Antonio,TX,Usa,Home Office


In [13]:
customer.to_csv('clean_customer.csv', index=False)
print('Saved clean_customer.csv')

Saved clean_customer.csv


# Product Table

In [14]:
product = pd.read_csv('Product_Dim.csv')
print(f'Shape: {product.shape}')
product.head()

Shape: (525, 5)


,Product_ID,Product_Name,Category,Sub_Category,Brand
0,1,Ergonomic Charger Max,Home Goods,Kitchenware,FashionForward
1,2,Wireless Charger Standard,Clothing,Apparel,TechGuru
2,3,Stylish Router Plus,Electronics,Laptops,HomeComfort
3,4,High-Performance Keyboard Mini,Electronics,Accessories,StyleSpot
4,5,Ergonomic Router Mini,Home Goods,kitchenware,FashionForward


In [15]:
print('Nulls:', product.isnull().sum().to_dict())
print(f'Full-row duplicates  : {product.duplicated().sum()}')
print(f'Product_ID duplicates: {product["Product_ID"].duplicated().sum()}')

Nulls: {'Product_ID': 0, 'Product_Name': 0, 'Category': 0, 'Sub_Category': 0, 'Brand': 0}
Full-row duplicates  : 25
Product_ID duplicates: 25


In [16]:
text_cols = ['Product_Name', 'Category', 'Sub_Category', 'Brand']
for col in text_cols:
    product[col] = product[col].astype(str).str.strip().str.title()

In [17]:
product['Sub_Category'] = product['Sub_Category'].replace({
    'Kitchen Ware' : 'Kitchenware',
    'kitchen ware' : 'Kitchenware',
    'KITCHENWARE'  : 'Kitchenware',
})

In [18]:
product = product.drop_duplicates(subset='Product_ID', keep='first')

print('Categories:', sorted(product['Category'].unique()))
print('Sub-categories:', sorted(product['Sub_Category'].unique()))
print(f'Clean shape: {product.shape}')

Categories: ['Books', 'Clothing', 'Electronics', 'Home Goods', 'Office Supplies']
Sub-categories: ['Accessories', 'Apparel', 'Appliances', 'Bedding', 'Decor', 'Fiction', 'Footwear', 'Furniture', 'Kitchenware', 'Laptops', 'Magazines', 'Non-Fiction', 'Paper', 'Pens', 'Printers', 'Smartphones', 'Textbooks', 'Wearables']
Clean shape: (500, 5)


In [19]:
print('Nulls:', product.isnull().sum().sum())
print('ID duplicates:', product['Product_ID'].duplicated().sum())
product.head(3)

Nulls: 0
ID duplicates: 0


,Product_ID,Product_Name,Category,Sub_Category,Brand
0,1,Ergonomic Charger Max,Home Goods,Kitchenware,Fashionforward
1,2,Wireless Charger Standard,Clothing,Apparel,Techguru
2,3,Stylish Router Plus,Electronics,Laptops,Homecomfort


In [20]:
product.to_csv('clean_product.csv', index=False)
print('Saved clean_product.csv')

Saved clean_product.csv


# Store Table

In [21]:
store = pd.read_csv('Store_Dim.csv')
print(f'Shape: {store.shape}')
store.head()

Shape: (52, 6)


,Store_ID,Store_Name,City,State,Region,Store_Type
0,1,Express Bazaar #1,San Diego,ca,South,Hypermarket
1,2,City Supermarket #2,San Antonio,TX,East,Supermarket
2,3,Express Supermarket #3,San Diego,CA,South,Department Store
3,4,Grand Store #4,Atlanta,GA,South,Convenience Store
4,5,Value Outlet #5,Boston,MA,Central,Specialty Store


In [22]:
print('Nulls:', store.isnull().sum().to_dict())
print(f'Full-row duplicates: {store.duplicated().sum()}')
print(f'Store_ID duplicates: {store["Store_ID"].duplicated().sum()}')

Nulls: {'Store_ID': 0, 'Store_Name': 0, 'City': 4, 'State': 4, 'Region': 0, 'Store_Type': 0}
Full-row duplicates: 2
Store_ID duplicates: 2


In [23]:
store['City']  = store['City'].fillna('Unknown')
store['State'] = store['State'].fillna('Unknown')

In [24]:
store['State']      = store['State'].str.strip().str.upper()
store['City']       = store['City'].str.strip().str.title()
store['Region']     = store['Region'].str.strip().str.title()
store['Store_Type'] = store['Store_Type'].str.strip().str.title()
store['Store_Name'] = store['Store_Name'].str.strip().str.title()

In [25]:
store = store.drop_duplicates(subset='Store_ID', keep='first')
print('Regions:', sorted(store['Region'].unique()))
print('Store types:', sorted(store['Store_Type'].unique()))
print(f'Clean shape: {store.shape}')

Regions: ['Central', 'East', 'South', 'West']
Store types: ['Convenience Store', 'Department Store', 'Hypermarket', 'Specialty Store', 'Supermarket']
Clean shape: (50, 6)


In [26]:
print('Nulls:', store.isnull().sum().sum())
print('ID duplicates:', store['Store_ID'].duplicated().sum())
store.head(3)

Nulls: 0
ID duplicates: 0


,Store_ID,Store_Name,City,State,Region,Store_Type
0,1,Express Bazaar #1,San Diego,CA,South,Hypermarket
1,2,City Supermarket #2,San Antonio,TX,East,Supermarket
2,3,Express Supermarket #3,San Diego,CA,South,Department Store


In [27]:
store.to_csv('clean_store.csv', index=False)
print('Saved clean_store.csv')

Saved clean_store.csv


# Employee Table

In [28]:
employee = pd.read_csv('Employee_Dim.csv')
print(f'Shape: {employee.shape}')
employee.head()

Shape: (210, 6)


,Employee_ID,Employee_Name,Gender,Position,Department,Hire_Date
0,1,Judy Jones,Female,NaN,Sales,2012-10-21
1,2,Charlie Garcia,Female,Supervisor,Human Resources,2019-06-13
2,3,Frank Jones,Male,Director,Human Resources,12/18/2013
3,4,Eve Rodriguez,Female,Specialist,Finance,2010-12-18
4,5,Alice Jones,Male,Specialist,Customer Service,04/16/2011


In [29]:
print('Nulls:', employee.isnull().sum().to_dict())
print(f'Full-row duplicates  : {employee.duplicated().sum()}')
print(f'Employee_ID duplicates: {employee["Employee_ID"].duplicated().sum()}')
print('\nDepartment raw values:')
print(employee['Department'].value_counts())

Nulls: {'Employee_ID': 0, 'Employee_Name': 0, 'Gender': 0, 'Position': 16, 'Department': 17, 'Hire_Date': 0}
Full-row duplicates  : 10
Employee_ID duplicates: 10

Department raw values:
Department
Marketing             24
Operations            23
Legal                 21
IT                    21
Sales                 17
Finance               17
Human Resources       16
Customer Service      15
human resources        6
 Finance               4
 Customer Service      4
sales                  4
 Legal                 3
 Human Resources       3
 IT                    3
finance                3
customer service       3
 Sales                 1
 Operations            1
operations             1
it                     1
marketing              1
 Marketing             1
Name: count, dtype: int64


In [30]:
employee['Position']   = employee['Position'].fillna('Unknown')
employee['Department'] = employee['Department'].fillna('Unknown')

In [31]:
text_cols = ['Employee_Name', 'Gender', 'Position', 'Department']
for col in text_cols:
    employee[col] = employee[col].astype(str).str.strip().str.title()

In [32]:
employee['Department'] = employee['Department'].str.replace(r'\bIt\b', 'IT', regex=True)

In [33]:
employee['Hire_Date'] = pd.to_datetime(employee['Hire_Date'], format='mixed', dayfirst=False)

In [34]:
employee['Hire_Year']   = employee['Hire_Date'].dt.year
employee['Tenure_Years'] = (pd.Timestamp('today') - employee['Hire_Date']).dt.days // 365

In [35]:
employee = employee.drop_duplicates(subset='Employee_ID', keep='first')


In [36]:
print('Department values after clean:')
print(employee['Department'].value_counts())
print(f'\nHire_Date range: {employee["Hire_Date"].min().date()} → {employee["Hire_Date"].max().date()}')
print(f'Clean shape: {employee.shape}')

Department values after clean:
Department
Marketing           26
IT                  25
Human Resources     24
Finance             23
Customer Service    22
Legal               22
Sales               21
Operations          21
Unknown             16
Name: count, dtype: int64

Hire_Date range: 2010-01-16 → 2023-12-20
Clean shape: (200, 8)


In [37]:
print('Nulls:', employee.isnull().sum().sum())
print('ID duplicates:', employee['Employee_ID'].duplicated().sum())
employee.head(3)

Nulls: 0
ID duplicates: 0


,Employee_ID,Employee_Name,Gender,Position,Department,Hire_Date,Hire_Year,Tenure_Years
0,1,Judy Jones,Female,Unknown,Sales,2012-10-21,2012,13
1,2,Charlie Garcia,Female,Supervisor,Human Resources,2019-06-13,2019,6
2,3,Frank Jones,Male,Director,Human Resources,2013-12-18,2013,12


In [38]:
employee.to_csv('clean_employee.csv', index=False)
print('Saved clean_employee.csv')

Saved clean_employee.csv


# Sales Fact Table

In [39]:
sales = pd.read_csv('Sales_Fact.csv')
print(f'Shape: {sales.shape}')
sales.head()

Shape: (561000, 11)


,Sale_ID,Customer_ID,Product_ID,Store_ID,Employee_ID,Sale_Date,Quantity,Unit_Price,Total_Price,Payment_Method,Shipping_Cost
0,1,954,164,13,93,2023-01-23,4,231.06,924.24,Cash,28.08
1,2,376,22,12,109,04/23/2020,1,24.80,24.80,Credit Card,27.50
2,3,559,197,31,53,2022-07-16,6,467.22,2803.32,Debit Card,15.77
3,4,803,452,20,56,28-Apr-21,4,212.47,849.88,Cash,10.60
4,5,238,241,42,84,2020-12-18,7,284.98,1994.86,Cash,22.01


In [40]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 561000 entries, 0 to 560999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Sale_ID         561000 non-null  int64  
 1   Customer_ID     561000 non-null  int64  
 2   Product_ID      561000 non-null  int64  
 3   Store_ID        561000 non-null  int64  
 4   Employee_ID     561000 non-null  int64  
 5   Sale_Date       561000 non-null  object 
 6   Quantity        561000 non-null  int64  
 7   Unit_Price      561000 non-null  float64
 8   Total_Price     561000 non-null  float64
 9   Payment_Method  561000 non-null  object 
 10  Shipping_Cost   516514 non-null  float64
dtypes: float64(3), int64(6), object(2)
memory usage: 47.1+ MB


In [41]:
nulls = sales.isnull().sum()
print(nulls[nulls > 0])
print(f'\nShipping null %: {sales["Shipping_Cost"].isnull().sum() / len(sales) * 100:.1f}%')

Shipping_Cost    44486
dtype: int64

Shipping null %: 7.9%


In [42]:
print(f'Full-row duplicates: {sales.duplicated().sum()}')
print(f'Sale_ID duplicates : {sales["Sale_ID"].duplicated().sum()}')

print('\n=== PAYMENT METHOD (raw) ===')
print(sales['Payment_Method'].value_counts())

Full-row duplicates: 11000
Sale_ID duplicates : 11000

=== PAYMENT METHOD (raw) ===
Payment_Method
Online Transfer        112179
Debit Card             112158
Credit Card            112117
Cash                   112052
  Online Transfer       14304
credit card             14257
  Cash                  14041
  Credit Card           14039
debit card              14035
cash                    13956
online transfer         13937
  Debit Card            13925
Name: count, dtype: int64


In [43]:
sales['Shipping_Cost'] = sales['Shipping_Cost'].fillna(0)

In [44]:
sales['Sale_Date'] = pd.to_datetime(sales['Sale_Date'], format='mixed', dayfirst=False)

In [45]:
sales['Payment_Method'] = sales['Payment_Method'].str.strip().str.title()

In [46]:
sales['Total_Price']   = sales['Total_Price'].round(2)
sales['Unit_Price']    = sales['Unit_Price'].round(2)
sales['Shipping_Cost'] = sales['Shipping_Cost'].round(2)

In [47]:
before = len(sales)
sales = sales.drop_duplicates()
print(f'Removed {before - len(sales):,} duplicate rows')

Removed 11,000 duplicate rows


In [48]:
sales['_calc'] = (sales['Quantity'] * sales['Unit_Price']).round(2)
mismatch = (abs(sales['Total_Price'] - sales['_calc']) > 0.01).sum()
print(f'Total_Price mismatches: {mismatch} (should be 0)')
sales = sales.drop(columns=['_calc'])

Total_Price mismatches: 0 (should be 0)


In [49]:
sales['Year']      = sales['Sale_Date'].dt.year
sales['Month']     = sales['Sale_Date'].dt.month
sales['Month_Name']= sales['Sale_Date'].dt.strftime('%b')
sales['Quarter']   = sales['Sale_Date'].dt.quarter
sales['Weekday']   = sales['Sale_Date'].dt.day_name()

In [50]:
print(f'\nPayment methods after clean: {sorted(sales["Payment_Method"].unique())}')
print(f'Date range: {sales["Sale_Date"].min().date()} → {sales["Sale_Date"].max().date()}')
print(f'Clean shape: {sales.shape}')


Payment methods after clean: ['Cash', 'Credit Card', 'Debit Card', 'Online Transfer']
Date range: 2020-01-01 → 2023-12-30
Clean shape: (550000, 16)


In [51]:
clean_cust = pd.read_csv('clean_customer.csv')
clean_prod = pd.read_csv('clean_product.csv')
clean_stor = pd.read_csv('clean_store.csv')
clean_emp  = pd.read_csv('clean_employee.csv')

before = len(sales)
sales = sales[sales['Customer_ID'].isin(clean_cust['Customer_ID'])]
sales = sales[sales['Product_ID'].isin(clean_prod['Product_ID'])]
sales = sales[sales['Store_ID'].isin(clean_stor['Store_ID'])]
sales = sales[sales['Employee_ID'].isin(clean_emp['Employee_ID'])]
print(f'Removed {before - len(sales):,} orphan rows (no matching dimension)')
print(f'Final Sales shape: {sales.shape}')

Removed 0 orphan rows (no matching dimension)
Final Sales shape: (550000, 16)


In [52]:
print('Nulls remaining:', sales.isnull().sum().sum())
print('ID duplicates  :', sales['Sale_ID'].duplicated().sum())
print('\nPayment value counts:')
print(sales['Payment_Method'].value_counts())
sales.head(3)

Nulls remaining: 0
ID duplicates  : 0

Payment value counts:
Payment_Method
Online Transfer    137674
Credit Card        137647
Debit Card         137401
Cash               137278
Name: count, dtype: int64


,Sale_ID,Customer_ID,Product_ID,Store_ID,Employee_ID,Sale_Date,Quantity,Unit_Price,Total_Price,Payment_Method,Shipping_Cost,Year,Month,Month_Name,Quarter,Weekday
0,1,954,164,13,93,2023-01-23,4,231.06,924.24,Cash,28.08,2023,1,Jan,1,Monday
1,2,376,22,12,109,2020-04-23,1,24.80,24.80,Credit Card,27.50,2020,4,Apr,2,Thursday
2,3,559,197,31,53,2022-07-16,6,467.22,2803.32,Debit Card,15.77,2022,7,Jul,3,Saturday


In [53]:
sales.to_csv('clean_sales.csv', index=False)
print('Saved → clean_sales.csv ✓')

Saved → clean_sales.csv ✓


In [54]:
files = {
    'clean_sales.csv'    : 'Sales',
    'clean_customer.csv' : 'Customer',
    'clean_employee.csv' : 'Employee',
    'clean_product.csv'  : 'Product',
    'clean_store.csv'    : 'Store',
}

print(f'{'Table':<12} {'Rows':>8} {'Cols':>5} {'Nulls':>7}')
print('-' * 38)
for f, name in files.items():
    df = pd.read_csv(f)
    print(f'{name:<12} {len(df):>8,} {df.shape[1]:>5} {df.isnull().sum().sum():>7}')

print('\nAll tables clean')

Table            Rows  Cols   Nulls
--------------------------------------
Sales         550,000    16       0
Customer        1,000     8       0
Employee          200     8       0
Product           500     5       0
Store              50     6       0

All tables clean
